In [1]:
import zipfile

import pandas as pd

from isochrone.download import RAW_DATA_DIR

FEED_PATH = RAW_DATA_DIR / "mvv_gtfs_2026-11-01.zip"
feed = zipfile.ZipFile(FEED_PATH)
feed.namelist()


['agency.txt',
 'calendar.txt',
 'calendar_dates.txt',
 'feed_info.txt',
 'routes.txt',
 'shapes.txt',
 'stops.txt',
 'stop_times.txt',
 'ticketing_deep_links.txt',
 'ticketing_identifiers.txt',
 'trips.txt']

In [2]:
from isochrone.gtfs import read_gtfs_table

read_gtfs_table(FEED_PATH, "feed_info")


,feed_publisher_name,feed_publisher_url,feed_lang,feed_start_date,feed_end_date,feed_version
0,Münchner VerkehrsVerbund (MVV),https://www.mvv-auskunft.de,DE,20261001,20261213,20261005


In [3]:
import datetime as dt

from isochrone.gtfs import active_service_ids

trips = read_gtfs_table(FEED_PATH, "trips", usecols=["trip_id", "service_id"])
trips_per_service = trips["service_id"].value_counts()

dates = pd.date_range("2026-11-02", "2026-12-13").date
counts = {
    date: trips_per_service.reindex(list(active_service_ids(FEED_PATH, date)), fill_value=0).sum()
    for date in dates
}
daily = pd.DataFrame({"trips": counts})
daily["weekday"] = [date.strftime("%a") for date in daily.index]
daily


,trips,weekday
2026-11-02,40830,Mon
2026-11-03,40830,Tue
2026-11-04,40814,Wed
2026-11-05,40807,Thu
2026-11-06,40980,Fri
2026-11-07,30588,Sat
2026-11-08,21381,Sun
2026-11-09,43605,Mon
2026-11-10,43543,Tue
2026-11-11,43522,Wed


In [4]:
from isochrone.gtfs import gtfs_time_to_seconds

stop_times = read_gtfs_table(
    FEED_PATH,
    "stop_times",
    usecols=["trip_id", "arrival_time", "departure_time", "stop_id", "stop_sequence"],
)
arr = gtfs_time_to_seconds(stop_times["arrival_time"])
dep = gtfs_time_to_seconds(stop_times["departure_time"])

pd.Series(
    {
        "rows": len(stop_times),
        "min departure (h)": round(dep.min() / 3600, 2),
        "max departure (h)": round(dep.max() / 3600, 2),
        "departures >= 24 h": (dep >= 24 * 3600).sum(),
        "dwell > 0": (dep > arr).sum(),
        "departure < arrival": (dep < arr).sum(),
    },
    dtype=object,
)


rows                   2217866
min departure (h)          0.0
max departure (h)        30.13
departures >= 24 h       77946
dwell > 0               288091
departure < arrival          0
dtype: object

In [5]:
from isochrone.connections import build_connections

TEST_DATE = dt.date(2026, 12, 9)

connections = build_connections(FEED_PATH, TEST_DATE)
duration_min = (connections["arr_time"] - connections["dep_time"]) / 60

print(len(connections), "connections from", connections["trip_id"].nunique(), "trips")
print((duration_min == 0).sum(), "connections with zero duration")
no_board = (~connections["can_board"]).sum()
no_alight = (~connections["can_alight"]).sum()
print(no_board, "no boarding,", no_alight, "no alighting")
duration_min.describe().round(1)


769125 connections from 41788 trips
7060 connections with zero duration
3340 no boarding, 388 no alighting


count    769125.0
mean          1.7
std           1.4
min           0.0
25%           1.0
50%           1.2
75%           2.0
max          53.0
dtype: float64

In [6]:
from isochrone.transfers import load_platforms

platforms = load_platforms(FEED_PATH)
print(len(platforms))
platforms[["x", "y"]].describe().round(0)


18861


,x,y
count,18861.0,18861.0
mean,701121.0,5336735.0
std,32491.0,31886.0
min,551176.0,5251164.0
25%,680841.0,5316596.0
50%,697663.0,5335605.0
75%,728333.0,5354535.0
max,802884.0,5576986.0


In [7]:
from isochrone.transfers import build_transfers

transfers = build_transfers(FEED_PATH)
print(len(transfers))
print(transfers["duration_s"].describe())

stop_names = read_gtfs_table(FEED_PATH, "stops", usecols=["stop_id", "stop_name"])
stop_names = stop_names.set_index("stop_id")["stop_name"]

from_marienplatz = transfers[transfers["from_stop"] == "de:09162:2:4:4"].copy()
from_marienplatz["to_name"] = from_marienplatz["to_stop"].map(stop_names)
from_marienplatz.sort_values("duration_s")


102831
count    102831.000000
mean        336.014743
std         186.199806
min         120.000000
25%         139.000000
50%         320.000000
75%         518.000000
max         833.000000
Name: duration_s, dtype: float64


,from_stop,to_stop,distance_m,duration_s,to_name
1620,de:09162:2:4:4,de:09162:2:4:4,0.000000,120,Marienplatz
26714,de:09162:2:4:4,de:09162:93:2:2,138.261096,300,Tal
26708,de:09162:2:4:4,de:09162:2:51:51,165.081041,335,Marienplatz U-Bahn Gleis 1 Ri Odeonsplatz
26706,de:09162:2:4:4,de:09162:2:41:81,198.341831,378,Marienplatz Gleis 1
26707,de:09162:2:4:4,de:09162:2:42:82,198.225933,378,Marienplatz Gleis 2
26709,de:09162:2:4:4,de:09162:2:52:52,246.450551,441,Marienplatz U-Bahn Gleis 2 Ri Sendlinger Tor
64152,de:09162:2:4:4,de:09162:112:5:5,260.908091,460,Viktualienmarkt
67307,de:09162:2:4:4,de:09162:17:2:1,325.408674,544,Nationaltheater
67241,de:09162:2:4:4,de:09162:165:1:1,338.408151,560,Schrannenhalle
67317,de:09162:2:4:4,de:09162:17:2:2,343.916367,568,Nationaltheater


In [8]:
import datetime as dt

from isochrone.connections import build_connections
from isochrone.csa import build_network, latest_departures

connections = build_connections(FEED_PATH, dt.date(2026, 12, 9))
network = build_network(connections, transfers)

arrival = 9 * 3600
targets = {"de:09184:460:11:51": arrival, "de:09184:460:11:52": arrival}
departures = latest_departures(network, targets)
print(len(departures))

result = pd.DataFrame({"stop_id": list(departures), "departure_s": list(departures.values())})
result["name"] = result["stop_id"].map(stop_names)
result["departure"] = pd.to_timedelta(result["departure_s"], unit="s")
result["minutes"] = (arrival - result["departure_s"]) / 60
stations = ["de:09162:2:", "de:09162:6:", "de:09162:70:"]
result[result["stop_id"].str.startswith(tuple(stations))].sort_values("stop_id")


8377


,stop_id,departure_s,name,departure,minutes
1562,de:09162:2:41:81,30060,Marienplatz Gleis 1,0 days 08:21:00,39.000000
1563,de:09162:2:42:82,30000,Marienplatz Gleis 2,0 days 08:20:00,40.000000
1564,de:09162:2:51:51,30860,Marienplatz U-Bahn Gleis 1 Ri Odeonsplatz,0 days 08:34:20,25.666667
1565,de:09162:2:52:52,30370,Marienplatz U-Bahn Gleis 2 Ri Sendlinger Tor,0 days 08:26:10,33.833333
2303,de:09162:6:2:1,29970,"Hauptbahnhof (U, Tram)",0 days 08:19:30,40.500000
2304,de:09162:6:2:2,30090,"Hauptbahnhof (U, Tram)",0 days 08:21:30,38.500000
2305,de:09162:6:2:5,29610,"Hauptbahnhof (U, Tram)",0 days 08:13:30,46.500000
2306,de:09162:6:40:81,30300,München Hbf Gleis 1,0 days 08:25:00,35.000000
2307,de:09162:6:40:82,29820,München Hbf Gleis 2,0 days 08:17:00,43.000000
2308,de:09162:6:51:51,29930,"Hauptbahnhof (U, Tram) U2/U8 Gleis 1 Ri Königs...",0 days 08:18:50,41.166667


In [9]:
import numpy as np

from isochrone.walking import find_nearby_platforms, to_projected

stops = read_gtfs_table(FEED_PATH, "stops", usecols=["stop_id", "stop_lat", "stop_lon"])
garching = stops.set_index("stop_id").loc["de:09184:460"]
dest_x, dest_y = to_projected(garching["stop_lon"], garching["stop_lat"])

egress = find_nearby_platforms(np.array([dest_x, dest_y]), platforms)
egress["name"] = egress["stop_id"].map(stop_names)
targets_point = dict(zip(egress["stop_id"], (arrival - egress["walk_s"]).tolist()))
departures_point = latest_departures(network, targets_point)
print(len(departures), "->", len(departures_point), "platforms reached")

both = pd.DataFrame({"before": departures, "after": departures_point})
both["gain_min"] = (both["after"] - both["before"]) / 60
both["name"] = both.index.map(stop_names)
print((both["gain_min"] > 0).sum(), "platforms leave later,", both["before"].isna().sum(), "new")
display(egress)
both.sort_values("gain_min", ascending=False).head(10)


8377 -> 8482 platforms reached
271 platforms leave later, 105 new


,point,stop_id,distance_m,walk_s,name
0,0,de:09184:460:30:1,9.307832,13,"Garch.,Forschungsz."
1,0,de:09184:460:11:52,11.660566,16,"Garching, Forschungszentrum Gleis 2"
2,0,de:09184:460:30:2,15.805689,21,"Garch.,Forschungsz."
3,0,de:09184:460:11:51,20.925050,28,"Garching, Forschungszentrum Gleis 1"
4,0,de:09184:460:30:3,40.172633,53,"Garch.,Forschungsz."
5,0,de:09184:460:31:4,156.690086,204,"Garch.,Forschungsz."
6,0,de:09184:2073:0:1,460.065152,599,"Garching, Boltzmannstraße"
7,0,de:09184:2070:0:1,517.974722,674,"Garching, Lichtenbergstraße"
8,0,de:09184:2071:0:1,763.070928,992,"Garching, Anna-Boyksen-Str."


,before,after,gain_min,name
de:09178:3583:0:1,25980.0,29580,60.0,"Hetzenhausen, Abzw."
de:09178:9606:0:1,25560.0,29160,60.0,"Neufahrn, Massenhausener Str."
de:09178:2776:0:2,26100.0,29700,60.0,"Massenhausen, Maibaum"
de:09178:2782:0:2,26220.0,29820,60.0,"Massenhausen, Raiffeisenkasse"
de:09178:2700:0:4,25320.0,28920,60.0,Bachenhausen
de:09178:9605:0:1,25440.0,29040,60.0,"Neufahrn, Volksfestplatz"
de:09178:2692:3:3,25620.0,28500,48.0,"Großnöbach, Kreuzung"
de:09178:2703:0:1,25560.0,28440,48.0,"Großnöbach, Kirche"
de:09177:3493:0:1,25380.0,27780,40.0,"Niederding, Ost"
de:09177:3495:0:1,25560.0,27960,40.0,"Niederding, St.-Martin-Straße"


In [11]:
from isochrone.maps import band_polygons, isochrone_map
from isochrone.walking import build_grid, latest_departures_from_points

grid = build_grid(platforms)
access = find_nearby_platforms(grid, platforms)
print(len(grid), "grid points,", len(access), "access pairs")

latest = latest_departures_from_points(
    grid, access, departures_point, np.array([dest_x, dest_y]), arrival
)
minutes = (arrival - latest) / 60
bands = band_polygons(grid, minutes)
display(bands[["low_min", "high_min", "n_cells"]])
from isochrone.download import RAW_DATA_DIR

isochrones = isochrone_map(bands, (float(garching["stop_lon"]), float(garching["stop_lat"])))
map_path = RAW_DATA_DIR.parent / "maps" / "garching_2026-12-09_0900.html"
map_path.parent.mkdir(exist_ok=True)
isochrones.save(map_path)
print(map_path)


925741 grid points, 5924978 access pairs


,low_min,high_min,n_cells
0,0.0,5.0,18
1,5.0,10.0,92
2,10.0,15.0,244
3,15.0,20.0,545
4,20.0,25.0,955
5,25.0,30.0,1437
6,30.0,35.0,2401
7,35.0,40.0,3719
8,40.0,45.0,4184
9,45.0,50.0,5131


/Users/francescocodeca/Desktop/project_maps/data/maps/garching_2026-12-09_0900.html
